In [ ]:
import __init__
import pandas as pd
from _utils._notebook_utils import get_stratified_metrics_xrd

#### 1st pass finetune - Mattergen XRD
- Dataset Source: [Mattergen Alex-MP-20](https://github.com/microsoft/mattergen/tree/main/data-release/alex-mp)
  - Columns: Database (manual) 
  - Reduced Formula (Source)
  - CIF (pmg - Cifwriter with symprec 0.1)
  - XRD 'Condition Vector' (with [_calculate_theor_XRD.py](_utils/_preprocessing/_calculate_theor_XRD.py))
    - pmg - XRDCalculator(wavelength="CuKa")
    - top 20 most intense peaks selected ($2\theta$ and int)
    - Normalisations
      - $2\theta$ min-max for 0,90
      - intensities min-max for 0,100
- Deduplicated
- Cleaned for CIF augmentation
  -  Note: I didnt filter to context length here because it was not implemented yet, but filter to context was flagged as True during model training which effectively does the same thing (less efficient)
- dataset pushed to HuggingFace as: c-bone/mattergen_XRD (90:10 train/valid sets)

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/mattergen_XRD-slider.jsonc'

#### 2nd pass finetune - CHILI-100K XRD
- Dataset Source: [CHILI-100K](https://github.com/UlrikFriisJensen/CHILI)
  - Columns: `Database`, `Material ID`, `Reduced Formula`, `CIF`
  - CIF source: generated structures by extracting structures from the dataset and then CifWriter with symprec 0.1
- Preprocessing and filtering
  - deduplicated the raw parquet
  - Added theoretical XRD peaks with [_calculate_theor_XRD.py](_utils/_preprocessing/_calculate_theor_XRD.py)
    - pmg - `XRDCalculator(wavelength="CuKa")`
    - top 20 most intense peaks selected ($2\theta$ and int)
    - Normalisations
      - $2\theta$ min-max for 0,90
      - intensities min-max for 0,100
  - Ran VUN metrics against `c-bone/lematerial_clean` and filtered to the 1536-token context limit
  - Built a leakproof split with mutually exclusive novelty tiers
    - test: 500 seen + 500 structurally novel + 500 compositionally novel
    - val: 1500 in-distribution structures
    - train: remaining rows
- Final dataset cleaned for CIF augmentation and pushed to HuggingFace as: `c-bone/chili100k_strat`

#### Making the dataset

In [ ]:
df = pd.read_parquet('/home/cyprien/Data_Gen/chili100k_data.parquet')
df.to_parquet('/home/cyprien/Data_Gen/chili100k_data.parquet', index=False)

In [ ]:
!python _utils/_preprocessing/_deduplicate.py \
  --input_parquet /home/cyprien/Data_Gen/chili100k_data.parquet \
  --output_parquet HF-databases/chili100k/chili100k_dedup.parquet

In [ ]:
!python _utils/_preprocessing/_calculate_theor_XRD.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --output_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --num_workers 16

In [ ]:
!python _utils/_metrics/VUN_metrics.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --huggingface_dataset "c-bone/lematerial_clean" \
    --load_processed_data "HF-databases/lematerial/lematerial_dedup.parquet" \
    --output_parquet HF-databases/chili100k/chili100k_dedup_vun.parquet \
    --output_csv HF-databases/chili100k/chili100k_dedup_vun.csv \
    --check_comp_novelty \
    --num_workers 32 \
    --bond_length_acceptability_cutoff 0.0

> Note: We remove invalid CIFs here, but set bond_length_acceptability_cutoff to 0.0 to avoid filtering out the experimentally observed materials that would fail this check.
> Otherwise, the goal here is to make sure no CIF info is leaked from any training phase to test phase apart from the 500 seen subset

The two cells below are just here to count the amount of tokens per CIF in order to remove the ones that would be above context. For more stats on the dataset see [Y_Dataset_stats.ipynb](notebooks/Y_Dataset_stats.ipynb):

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup_vun.parquet \
    --output_parquet HF-databases/chili100k/chili100k_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
df_count = pd.read_parquet('HF-databases/chili100k/chili100k_count.parquet')
df_dedup = pd.read_parquet('HF-databases/chili100k/chili100k_dedup_vun.parquet')

df_merged = pd.merge(df_dedup, df_count[['Material ID', 'token_count']], on='Material ID', how='inner')
df_filtered = df_merged[df_merged['token_count'] <= 1536]
df_filtered.to_parquet('HF-databases/chili100k/chili100k_dedup_vun_filtered.parquet', index=False)

Now, from the tilered dataset, we make the stratified splits

In [ ]:
df = pd.read_parquet('HF-databases/chili100k/chili100k_dedup_vun_filtered.parquet')

df = df[~((df['is_valid'] == False) & (df['is_unique'] == False))]
df = df.drop(columns=['is_valid', 'is_unique'])

# Define strictly mutually exclusive masks to prevent data overlap
# mask_comp_novel = df['is_comp_novel'] == True and 'Reduced Formula' is unique in the dataset
# We want to ensure that if a composition is marked as novel, it does not have duplicates in the dataset, which could leak information across splits. Therefore, we check for both conditions: is_comp_novel is True and the Reduced Formula is not duplicated (i.e., unique).
mask_comp_novel = (df['is_comp_novel'] == True) & (df['Reduced Formula'].duplicated(keep=False) == False)
print(f"Number of comp novel structures: {mask_comp_novel.sum()}")
mask_struct_novel = (df['is_novel'] == True) & (df['is_comp_novel'] == False)
mask_in_dist = df['is_novel'] == False 

# Sample the test set
test_comp = df[mask_comp_novel].sample(n=500, random_state=1)
test_struct = df[mask_struct_novel].sample(n=500, random_state=1)
test_in_dist = df[mask_in_dist].sample(n=500, random_state=1)
test_indices = pd.concat([test_comp, test_struct, test_in_dist]).index

# Filter out the test set to define the remaining pool
remaining_df = df.drop(test_indices)

# Sample the validation set (1500 from in-distribution structures)
val_indices = remaining_df[remaining_df['is_novel'] == False].sample(n=1500, random_state=1).index

df['Split'] = 'train'
df.loc[val_indices, 'Split'] = 'val'
df.loc[test_indices, 'Split'] = 'test'

print(df['Split'].value_counts())

# renam 'Generated CIF' to CIF
df = df.rename(columns={'Generated CIF': 'CIF'})
df.to_parquet('HF-databases/chili100k/chili100k_dedup_leakproof.parquet', index=False)

Clean it up before shipping to Hugging-Face

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet 'HF-databases/chili100k/chili100k_clean.parquet' \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_save_dataset_to_HF.py \
    --input_parquet 'HF-databases/chili100k/chili100k_clean.parquet' \
    --output_parquet 'HF-databases/chili100k_strat.parquet' \
    --save_hub

### Training

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/chili100k-xrd-slider-opt.jsonc'

> This model loads weights from the 1st pass as show in top of the notebook

### Generating

First create prompts from the held-out CHILI-100K test split, then run repeated generations for the conditioned and text-only baselines.

In [ ]:
!python _utils/_generating/make_prompts.py \
    --HF_dataset 'c-bone/chili100k_strat' \
    --split 'test' \
    --automatic \
    --output_parquet '_artifacts/chili-xrd/chili-test_prompts.parquet' \
    --level 'level_3' \
    --condition_columns 'condition_vector'

#### Generate materials from the Chili model and XRD information (Repeated 3x)

In [ ]:
!python _utils/_generating/generate_CIFs.py --config '_config_files/generation/conditional/xrd_studies/chili-xrd_eval.jsonc'

In [ ]:
!python _utils/_generating/postprocess.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_gen.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_post.parquet' \
    --num_workers 32 \
    --column_name 'Generated CIF'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_gen.parquet' \
    --num_gens 20 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_gen.parquet' \
    --num_gens 1 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-1perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

#### Comparison to text-only model

We mirror the exact same training pipeline as above, however here we never let the model see XRD info in any of the training phase to understand the gains from XRD conditioning across the board

> **Note**: This comparison keeps the same held-out CHILI-100K prompt split, but switches to the unconditional 2-pass finetuning so the effect of the XRD channel can be isolated.

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/mattergen_XRD-uncond.jsonc'

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/chili100k-uncond-opt.jsonc'

In [ ]:
!python _utils/_generating/generate_CIFs.py --config '_config_files/generation/conditional/xrd_studies/chili-xrd-uncond_eval.jsonc'

In [ ]:
!python _utils/_generating/postprocess.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_gen.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_post.parquet' \
    --num_workers 32 \
    --column_name 'Generated CIF'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_gen.parquet' \
    --num_gens 20 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_gen.parquet' \
    --num_gens 1 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

### Aggregate Results

Aggregate the repeated conditioned and text-only runs, then report stratified averages with standard errors across the three repeats.

In [ ]:
paths = {
    'cond-20perp': '_artifacts/chili-xrd/chili-ft-20perp-test_metrics1.parquet',
    'cond-20perp-v2': '_artifacts/chili-xrd/chili-ft-20perp-test_metrics2.parquet',
    'cond-20perp-v3': '_artifacts/chili-xrd/chili-ft-20perp-test_metrics3.parquet',
    'cond-1perp': '_artifacts/chili-xrd/chili-ft-1perp-test_metrics1.parquet',
    'cond-1perp-v2': '_artifacts/chili-xrd/chili-ft-1perp-test_metrics2.parquet',
    'cond-1perp-v3': '_artifacts/chili-xrd/chili-ft-1perp-test_metrics3.parquet',
    'uncond-20perp': '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics1.parquet',
    'uncond-20perp-v2': '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics2.parquet',
    'uncond-20perp-v3': '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics3.parquet',
    'uncond-1perp': '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics1.parquet',
    'uncond-1perp-v2': '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics2.parquet',
    'uncond-1perp-v3': '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics3.parquet',
}

tier_sizes = {
    'Overall': 1500,
    'Memorized (Seen Comp & Struct)': 500,
    'Structurally Novel (Seen Comp)': 500,
    'Compositionally Novel (Unseen Comp)': 500,
}

tables = {}
for run_name, parquet_path in paths.items():
    df_metrics = pd.read_parquet(parquet_path)
    tables[run_name] = get_stratified_metrics_xrd(
        df_metrics,
        n_test=tier_sizes,
        only_matched=False,
        verbose=False,
    )

final_table = pd.concat(tables, names=['run', 'tier'])
final_table.index = [f'{run}__{tier}' for run, tier in final_table.index]
final_table.to_parquet('_artifacts/chili-xrd/chili-slider-vs-uncond-all-results.parquet')
final_table

In [ ]:
final_table = pd.read_parquet('_artifacts/chili-xrd/chili-slider-vs-uncond-all-results.parquet')

base_conditions = {}
for index in final_table.index:
    if '-v2' in index:
        base_name = index.replace('-v2', '')
    elif '-v3' in index:
        base_name = index.replace('-v3', '')
    else:
        base_name = index
    
    if base_name not in base_conditions:
        base_conditions[base_name] = []
    base_conditions[base_name].append(index)

# averaged results with standard error between the 3 runs
averaged_results = {}
stderr_results = {}
for base_name, variants in base_conditions.items():
    variant_data = []
    for variant in variants:
        if variant in final_table.index:
            variant_data.append(final_table.loc[variant])
    
    if variant_data:
        data_df = pd.concat(variant_data, axis=1)
        averaged_results[base_name] = data_df.mean(axis=1)
        stderr_results[base_name] = data_df.std(axis=1) / np.sqrt(len(variant_data))

averaged_table = pd.DataFrame.from_dict(averaged_results, orient='index')
stderr_table = pd.DataFrame.from_dict(stderr_results, orient='index')

formatted_table = pd.DataFrame(index=averaged_table.index, columns=averaged_table.columns, dtype=object)
for col in averaged_table.columns:
    for idx in averaged_table.index:
        mean_val = averaged_table.loc[idx, col]
        stderr_val = stderr_table.loc[idx, col]
        if pd.isna(stderr_val) or stderr_val == 0:
            formatted_table.loc[idx, col] = f"{mean_val:.3f}"
        else:
            formatted_table.loc[idx, col] = f"{mean_val:.3f} (±{stderr_val:.3f})"

formatted_table.to_csv('_artifacts/chili-xrd/chili-slider-vs-uncond-all-results_formatted.csv')

> Note: the results here differ very slightly from any pooled plots because this table averages each repeated run first and then reports the associated stderr, rather than concatenating all rows across repeats before computing summary statistics.